# F-07: Tower 2 — fixing the broken evaluation (TOWER 2 ONLY)

**Scope: Tower 2 only.** Tower 2 (Red farmlet, 'Great Field' / Catchment 2) is the data-poor tower: EC CH4 exists
**only Oct 2017 – Jun 2019** (grassland period; the CH4 analyser was relocated to Tower 9 in Jul 2019 when the Red
farmlet went arable). All prior results for Tower 2 were strongly **negative** (R-01 RF = −16.9; F-06 = −0.045).

**Root cause (this experiment):** it was a **broken evaluation, not a data/model failure.** Catchment 2 had ~10 cattle
in 2018 (FCH4 mean ≈ 42) but **zero livestock in early 2019** (FCH4 mean ≈ 2) — animals removed before the arable
conversion. The D-15 split (train 2018 / test Jan–May 2019) trains on the high-flux regime and tests on the
near-zero regime → catastrophic R². Gap-filling is **interpolation**, so the correct evaluation is a **full-period
gap-CV** (mask gaps anywhere across 2017–2019, fill from surrounding data), not a year split.

Tests (F-06 best feature set: gap-filled met + FCO2 + livestock density + lags + GPP + management):
- **MDS** and **RFm** under the **D-15 year split** (reproduce the broken result)
- **MDS**, **RFm solo**, **RFm pooled (T2+T4+T9)** under **full-period gap-CV** (the fix)
Reports median R² per gap scenario + **improvement over MDS** (the supervisor-endorsed framing for an open system).

In [1]:
from pathlib import Path
import datetime
import numpy as np, pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import r2_score
HOURLY=Path("../../data/Hourly"); RESULTS=Path("../../results")
PLAUS_LOW,PLAUS_HIGH=-500,3000
N_REPS,MASK_FRAC=5,0.25
SCENARIOS={"vs":1,"s":4,"m":32,"l":288,"m1":"mixed"}
LSU={"cattle":1.0,"sheep":0.1,"lamb":0.05}; AUX=["_hs","_hc","_ds","_dc"]; LAG_HOURS=[168,336,504,672]
AREA={2:6.65,4:7.75,9:7.75}; C4="Catchment 4 After  2013/08/13"
# Tower 2 valid CH4 period (grassland): Oct 2017 - Jun 2019
T2_DOMAIN=(pd.Timestamp("2017-10-01"), pd.Timestamp("2019-06-30"))

## 1  Config + F-06 feature builder

In [2]:
def cfg(t):
    cat=C4 if t==4 else f"Catchment {t}"
    met=[f"SWIN_1_1_1 [Tower {t}]",f"TA_0_0_1 [Tower {t}]",f"VPD_0_0_1 [Tower {t}]",f"PPFD_1_1_1 [Tower {t}]",
         f"RN_1_1_1 [Tower {t}]",f"WS_0_0_1 [Tower {t}]",f"USTAR_0_0_1 [Tower {t}]",f"SHF_1_1_1 [Tower {t}]",
         f"Precipitation (mm) [{cat}]","TS_1_1_1 [Tower 9]",f"Soil Moisture @ 10cm Depth (%) [{cat}]"]
    return dict(t=t,cat=cat,area=AREA[t],tgt=f"FCH4_1_1_1 [Tower {t}]",ssitc=f"FCH4_SSITC_TEST_1_1_1 [Tower {t}]",
        sw=f"SWIN_1_1_1 [Tower {t}]",ta=f"TA_0_0_1 [Tower {t}]",met=met,fc=f"FC_1_1_1 [Tower {t}]",
        gpp=f"GPP [Tower {t}]",reco=f"Reco [Tower {t}]",swc=f"Soil Moisture @ 10cm Depth (%) [{cat}]",
        liv={s:f"{s}_{cat}" for s in LSU},mc=f"mgmt_t{t}_cut_recency",mm=f"mgmt_t{t}_manure_recency")
CFG={t:cfg(t) for t in [2,4,9]}

## 2  Load (consolidated + gap-filled FCO2 + management + REddyProc met/GPP)

In [3]:
df=pd.read_csv(HOURLY/"consolidated_hourly.csv",low_memory=False)
df["Datetime"]=pd.to_datetime(df["Datetime"],format="mixed"); df=df.set_index("Datetime")
for f in ["fco2_gapfilled.csv","management_features.csv","reddyproc_processed.csv"]:
    e=pd.read_csv(HOURLY/f,low_memory=False); e["Datetime"]=pd.to_datetime(e["Datetime"],format="mixed")
    df=df.join(e.set_index("Datetime"),how="left")
for t in [2,4,9]: df[f"FC_1_1_1 [Tower {t}]"]=df[f"FC_gapfilled [Tower {t}]"]
print("loaded",df.shape)

loaded (70153, 522)


## 3  Functions (calendar gaps over a domain, MDS, feature frame)

In [4]:
def insert_calendar_gaps(df_qc, target, domain_mask, gap_hours, n_reps=N_REPS, seed=0):
    """Mask fixed-length CALENDAR gaps over a contiguous hourly domain (full index, incl. NaN),
    until ~MASK_FRAC of valid obs in the domain are covered. Matches F-01..F-06 semantics."""
    dom_ts=df_qc.index[domain_mask]; valid=df_qc.loc[domain_mask,target].notna().values
    n=len(dom_ts); target_n=max(1,int(valid.sum()*MASK_FRAC)); rb=np.random.default_rng(seed); reps=[]
    for _ in range(n_reps):
        rng=np.random.default_rng(int(rb.integers(0,2**31))); occ=np.zeros(n,bool); m=0
        for sp in rng.permutation(n):
            if m>=target_n: break
            gh=int(rng.choice([1,4,32,288])) if gap_hours=="mixed" else gap_hours; ep=min(int(sp)+gh,n)
            if occ[sp:ep].any(): continue
            occ[sp:ep]=True; m+=int(valid[sp:ep].sum())
        reps.append(dom_ts[occ & valid])
    return reps

def mds_fill_batch(df_obs, target, sw_col, ta_col, gap_ts):
    SW_TOL,TA_TOL=50.0,2.5; WINDOWS=[pd.Timedelta(days=d) for d in [7,14,28,91]]
    av=df_obs[df_obs[target].notna()]; ay=av[target].values.astype(float)
    ahr=av.index.hour.to_numpy(); adoy=av.index.dayofyear.to_numpy(); ats=av.index.to_numpy()
    asw=av[sw_col].values.astype(float); ata=av[ta_col].values.astype(float)
    gi=pd.DatetimeIndex(gap_ts); gsw=df_obs.reindex(gi)[sw_col].values.astype(float); gta=df_obs.reindex(gi)[ta_col].values.astype(float)
    preds={}
    for i,t in enumerate(gap_ts):
        tt=np.datetime64(t); hr=t.hour; doy=t.dayofyear; sv=gsw[i]; tv=gta[i]
        day=(not np.isnan(sv)) and sv>10.0; filled=False
        for wd in WINDOWS:
            w=wd.to_timedelta64(); m=(ats>=tt-w)&(ats<=tt+w)&(np.abs(ahr-hr)<=1)
            if not np.isnan(tv): m&=(np.abs(ata-tv)<=TA_TOL)|np.isnan(ata)
            if day and not np.isnan(sv): m&=(np.abs(asw-sv)<=SW_TOL)|np.isnan(asw)
            c=ay[m]
            if len(c)>=1: preds[t]=float(np.nanmean(c)); filled=True; break
        if not filled:
            sh=np.abs(ahr-hr)<=1; dd=np.minimum(np.abs(adoy-doy),365-np.abs(adoy-doy)); c=ay[sh&(dd<=7)]
            if len(c)>=1: preds[t]=float(np.nanmean(c))
    return preds

def frame(t, pooled):
    c=CFG[t]; d=df.copy(); tgt=c["tgt"]
    d.loc[~d[c["ssitc"]].isin([0,1]),tgt]=np.nan
    d.loc[d[tgt].notna()&~d[tgt].between(PLAUS_LOW,PLAUS_HIGH,inclusive="both"),tgt]=np.nan
    h,doy=d.index.hour,d.index.dayofyear
    d["_hs"]=np.sin(2*np.pi*h/24);d["_hc"]=np.cos(2*np.pi*h/24);d["_ds"]=np.sin(2*np.pi*doy/365);d["_dc"]=np.cos(2*np.pi*doy/365)
    g=pd.DataFrame(index=d.index); g["target"]=d[tgt]
    for k in c["met"]:
        nm=k.split(" [")[0]; g[nm]= d[k+"__f"] if (k+"__f") in d.columns else d[k]
    g["fc"]=d[c["fc"]]
    for a in AUX: g[a]=d[a]
    lsu=sum(d[c["liv"][s]].fillna(0)*w for s,w in LSU.items())
    g["lsu_dens"]=lsu/c["area"]; g["graze"]=(sum(d[c["liv"][s]].fillna(0) for s in LSU)>0).astype(float)
    swc=d[c["swc"]+"__f"] if (c["swc"]+"__f") in d.columns else d[c["swc"]]
    ts=d["TS_1_1_1 [Tower 9]__f"] if "TS_1_1_1 [Tower 9]__f" in d.columns else d["TS_1_1_1 [Tower 9]"]
    for lag in LAG_HOURS: g[f"swc_l{lag}"]=swc.shift(lag); g[f"ts_l{lag}"]=ts.shift(lag)
    g["mgmt_cut"]=d[c["mc"]]; g["mgmt_manure"]=d[c["mm"]]; g["gpp"]=d[c["gpp"]]; g["reco"]=d[c["reco"]]
    if pooled:
        for tt in [2,4,9]: g[f"is_t{tt}"]=1.0 if tt==t else 0.0
    g["_y"]=d.index.year
    return g

FEAT=[m.split(" [")[0] for m in CFG[2]["met"]]+["fc"]+AUX+["lsu_dens","graze"]+[f"swc_l{l}" for l in LAG_HOURS]+[f"ts_l{l}" for l in LAG_HOURS]+["mgmt_cut","mgmt_manure","gpp","reco"]
DUM=["is_t2","is_t4","is_t9"]

## 4  Run variants (Tower 2)

In [5]:
g2=frame(2,False)
idx=df.index
DOM=(idx>=T2_DOMAIN[0])&(idx<=T2_DOMAIN[1])          # full-period domain
Y2019=(idx.year==2019)                                # D-15 test domain
nvalid=int((DOM & g2["target"].notna().values).sum())
print(f"Tower 2 valid in full-period domain: {nvalid}")

def fit(feat,trd):
    imp=SimpleImputer(strategy="mean"); rf=RandomForestRegressor(n_estimators=500,min_samples_leaf=5,n_jobs=-1,random_state=42)
    rf.fit(imp.fit_transform(trd[feat].values),trd["target"].values); return rf,imp

def run_mds(domain_mask):
    c=CFG[2]; d=df.copy(); tgt=c["tgt"]
    d.loc[~d[c["ssitc"]].isin([0,1]),tgt]=np.nan; d.loc[d[tgt].notna()&~d[tgt].between(PLAUS_LOW,PLAUS_HIGH,inclusive="both"),tgt]=np.nan
    out={}
    for sc,gh in SCENARIOS.items():
        r=[]
        for gt in insert_calendar_gaps(d,tgt,domain_mask,gh):
            if len(gt)<5: continue
            sav=d.loc[gt,tgt].copy(); d.loc[gt,tgt]=np.nan
            p=mds_fill_batch(d,tgt,c["sw"],c["ta"],list(gt)); d.loc[gt,tgt]=sav
            ts=[t for t in gt if t in p]
            if len(ts)>=5: r.append(r2_score(sav.loc[ts].values,[p[t] for t in ts]))
        out[sc]=np.median(r) if r else np.nan
    return out

def run_rf(mode):
    pooled=(mode=="pool"); feat=FEAT+DUM if pooled else FEAT
    frames={t:frame(t,pooled) for t in ([2,4,9] if pooled else [2])}; g=frames[2]; out={}
    for sc,gh in SCENARIOS.items():
        r=[]
        dommask = Y2019 if mode=="d15" else DOM
        for gt in insert_calendar_gaps(g,"target",dommask,gh):
            if len(gt)<5: continue
            if mode=="d15":
                trd=g[(g["_y"]==2018)&g["target"].notna()]
            else:
                base=g[DOM & g["target"].notna().values]; trd=base.drop(index=gt,errors="ignore")
                if pooled:
                    trd=pd.concat([trd]+[frames[t][frames[t]["_y"].isin(range(2018,2022))&frames[t]["target"].notna()] for t in [4,9]],ignore_index=True)
            rf,imp=fit(feat,trd); yp=rf.predict(imp.transform(g.loc[gt,feat].values))
            r.append(r2_score(g.loc[gt,"target"].values,yp))
        out[sc]=np.median(r) if r else np.nan
    return out

results={"MDS_D15":run_mds(Y2019),"RFm_solo_D15":run_rf("d15"),
         "MDS_fullCV":run_mds(DOM),"RFm_solo_fullCV":run_rf("fullCV"),"RFm_pool_fullCV":run_rf("pool")}
print("done")

Tower 2 valid in full-period domain: 4890


done


## 5  Results — Tower 2 R² and improvement over MDS

In [6]:
ORDER=["MDS_D15","RFm_solo_D15","MDS_fullCV","RFm_solo_fullCV","RFm_pool_fullCV"]
tab=pd.DataFrame(results).T.reindex(ORDER)[list(SCENARIOS)].round(3)
print("=== Tower 2 median R2 by gap scenario ===\n"+tab.to_string())
print("\nOverall median R2:\n"+tab.median(axis=1).round(3).to_string())
print("\n=== Improvement over MDS (full-period CV) ===")
print((tab - tab.loc["MDS_fullCV"]).round(3).loc[["RFm_solo_fullCV","RFm_pool_fullCV"]].to_string())

=== Tower 2 median R2 by gap scenario ===
                    vs      s      m      l     m1
MDS_D15          0.043 -0.162 -0.127 -0.544 -0.216
RFm_solo_D15    -0.062 -0.078 -0.089 -0.105 -0.083
MDS_fullCV      -0.417 -0.488 -0.427 -0.711 -1.007
RFm_solo_fullCV  0.395  0.522  0.511  0.355  0.324
RFm_pool_fullCV  0.519  0.657  0.618  0.490  0.472

Overall median R2:
MDS_D15           -0.162
RFm_solo_D15      -0.083
MDS_fullCV        -0.488
RFm_solo_fullCV    0.395
RFm_pool_fullCV    0.519

=== Improvement over MDS (full-period CV) ===
                    vs      s      m      l     m1
RFm_solo_fullCV  0.812  1.010  0.938  1.066  1.331
RFm_pool_fullCV  0.936  1.145  1.045  1.201  1.479


## 6  Append to benchmarks.csv

In [7]:
bench=RESULTS/"benchmarks.csv"; today=datetime.date.today().isoformat()
ex=pd.read_csv(bench); ex=ex[ex["replication"]!="F-07"]
rows=[]
for variant,scn in results.items():
    for sc,v in scn.items():
        if pd.isna(v): continue
        rows.append({"replication":"F-07","model":"RFm" if "RFm" in variant else "MDS","tower":"Tower 2",
            "feature_set":variant,"scenario":sc,"split":"fullCV" if "fullCV" in variant else "D15",
            "R2":round(float(v),4),"date":today,"notes":"F07 Tower2-only: full-period gap-CV vs D-15 split; F-06 features (D-34)"})
new=pd.DataFrame(rows); comb=pd.concat([ex,new],ignore_index=True); comb.to_csv(bench,index=False)
print(f"Wrote {len(new)} F-07 rows. Total {len(comb)}.")

Wrote 25 F-07 rows. Total 2765.
